# EDA - Bộ dữ liệu dự đoán tiểu đường

Nguồn: diabetes_prediction_dataset.csv (Kaggle - Mohammed Mustafa). 100.000 dòng, 8 đặc trưng và nhãn diabetes.
Notebook khảo sát dữ liệu và chốt cách tiền xử lý. Giải thích chi tiết xem mo_ta.txt.


In [14]:
import csv, os

CSV_PATH = os.path.join("data", "diabetes_prediction_dataset.csv")

def doc_du_lieu(path):
    with open(path, encoding="utf-8") as f:
        return list(csv.DictReader(f))

def ti_le_mac(records):
    if not records:
        return 0.0
    return 100.0 * sum(1 for r in records if r["diabetes"] == "1") / len(records)

def in_bang(tieu_de, cac_nhom):
    print(tieu_de)
    for ten, nhom in cac_nhom:
        print(f"  {ten:<12}: n={len(nhom):>6}  ty le mac={ti_le_mac(nhom):>6.2f}%")

rows = doc_du_lieu(CSV_PATH)
len(rows)

100000

## 1. Tổng quan

In [15]:
print("So dong:", len(rows))
print("Cot    :", list(rows[0].keys()))
print("O trong:", sum(1 for r in rows for v in r.values() if not v.strip()))

So dong: 100000
Cot    : ['gender', 'age', 'hypertension', 'heart_disease', 'smoking_history', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'diabetes']
O trong: 0


In [16]:
# So gia tri phan biet cua tung cot -> phat hien cot co qua it gia tri so voi 100k dong
for c in rows[0].keys():
    print(f"{c:<22}: {len(set(r[c] for r in rows)):>6} gia tri phan biet")

gender                :      3 gia tri phan biet
age                   :    102 gia tri phan biet
hypertension          :      2 gia tri phan biet
heart_disease         :      2 gia tri phan biet
smoking_history       :      6 gia tri phan biet
bmi                   :   4247 gia tri phan biet
HbA1c_level           :     18 gia tri phan biet
blood_glucose_level   :     18 gia tri phan biet
diabetes              :      2 gia tri phan biet


In [17]:
import statistics as th

def mo_ta(col):
    v = sorted(float(r[col]) for r in rows)
    print(f"{col:<20} min={v[0]:>7.2f}  max={v[-1]:>7.2f}  mean={th.mean(v):>7.2f}  "
          f"median={th.median(v):>7.2f}  sd={th.stdev(v):>6.2f}")

for c in ["age", "bmi", "HbA1c_level", "blood_glucose_level"]:
    mo_ta(c)

age                  min=   0.08  max=  80.00  mean=  41.89  median=  43.00  sd= 22.52
bmi                  min=  10.01  max=  95.69  mean=  27.32  median=  27.32  sd=  6.64
HbA1c_level          min=   3.50  max=   9.00  mean=   5.53  median=   5.80  sd=  1.07
blood_glucose_level  min=  80.00  max= 300.00  mean= 138.06  median= 140.00  sd= 40.71


Ba điểm bất thường cần điều tra ở mục 3: tuổi nhỏ nhất là 0.08 (không phải 0);
BMI có mean và median trùng khớp tuyệt đối (27.32); HbA1c và blood_glucose_level
chỉ có 18 giá trị phân biệt dù có 100.000 dòng, khác hẳn age/bmi có hàng nghìn
giá trị phân biệt.

## 2. Phân bố nhãn

In [18]:
n = len(rows)
mac = sum(1 for r in rows if r["diabetes"] == "1")
print(f"Khong mac: {n-mac:>6}  ({100.0*(n-mac)/n:.2f}%)")
print(f"Mac      : {mac:>6}  ({100.0*mac/n:.2f}%)")

Khong mac:  91500  (91.50%)
Mac      :   8500  (8.50%)


Dữ liệu mất cân bằng 91.5/8.5. Đánh giá bằng precision, recall, F1 và mốc đoán đa số; không dùng riêng accuracy.


## 3. Chất lượng dữ liệu

In [19]:
cols = list(rows[0].keys())
seen, trung = set(), 0
for r in rows:
    k = tuple(r[c] for c in cols)
    if k in seen:
        trung += 1
    else:
        seen.add(k)
print("Trung lap        :", trung, "-> con", len(rows) - trung)

Trung lap        : 3854 -> con 96146


Xử lý:
Trùng lặp 3.854 dòng: loại, còn 96.146.

### age < 1

Mục 1: age min = 0.08.

In [20]:
tuoi_nho = [r for r in rows if float(r["age"]) < 1]
gia_tri = sorted(set(float(r["age"]) for r in tuoi_nho))
print("So dong age<1        :", len(tuoi_nho))
print("Cac gia tri (thang):  ", gia_tri)
print("So mac trong nhom     :", sum(1 for r in tuoi_nho if r["diabetes"] == "1"))
print("So co cao huyet ap    :", sum(1 for r in tuoi_nho if r["hypertension"] == "1"))
print("So co benh tim        :", sum(1 for r in tuoi_nho if r["heart_disease"] == "1"))

So dong age<1        : 911
Cac gia tri (thang):   [0.08, 0.16, 0.24, 0.32, 0.4, 0.48, 0.56, 0.64, 0.72, 0.8, 0.88]
So mac trong nhom     : 0
So co cao huyet ap    : 0
So co benh tim        : 0


Xử lý:
Giữ. 11 giá trị cách đều 0.08 (~1/12) là tuổi tính theo tháng, không phải lỗi nhập liệu. Cả 911 dòng đều không mắc bệnh, không cao huyết áp, không bệnh tim. Loại là can thiệp chủ quan không căn cứ.

### bmi = 27.32 lặp bất thường

Mục 1: mean = median = 27.32.

In [21]:
from collections import Counter

dem_bmi = Counter(float(r["bmi"]) for r in rows)
gia_tri_pho_bien, tan_suat = dem_bmi.most_common(1)[0]
print(f"Gia tri bmi xuat hien nhieu nhat: {gia_tri_pho_bien} ({tan_suat} dong, "
      f"{100.0*tan_suat/len(rows):.2f}%)")

nhom_dien_khuyet = [r for r in rows if float(r["bmi"]) == gia_tri_pho_bien]
nhom_con_lai = [r for r in rows if float(r["bmi"]) != gia_tri_pho_bien]
print(f"Ty le mac trong nhom bmi={gia_tri_pho_bien}: {ti_le_mac(nhom_dien_khuyet):.2f}%")
print(f"Ty le mac o cac dong con lai       : {ti_le_mac(nhom_con_lai):.2f}%")

Gia tri bmi xuat hien nhieu nhat: 27.32 (25495 dong, 25.50%)
Ty le mac trong nhom bmi=27.32: 6.01%
Ty le mac o cac dong con lai       : 9.35%


Xử lý:
Giữ, ghi nhận hạn chế. 25.495 dòng (25,50%) có bmi = 27.32, đúng mean của cột: dữ liệu bị điền khuyết bằng mean, không phải BMI đo thật. Tỷ lệ mắc trong nhóm này 6,01% so với 9,35% phần còn lại - bucket BMI 25-29.9 và hệ số BMI ở Job 2 bị pha loãng. Không điền lại vì không khôi phục được BMI thật.

### HbA1c và glucose chỉ 18 giá trị

Mục 1: mỗi cột 18 giá trị phân biệt trên 100.000 dòng.

In [22]:
print("HbA1c_level        :", sorted(set(float(r["HbA1c_level"]) for r in rows)))
print("blood_glucose_level:", sorted(set(float(r["blood_glucose_level"]) for r in rows)))
print("So gia tri phan biet cua age (doi chieu):", len(set(float(r["age"]) for r in rows)))

HbA1c_level        : [3.5, 4.0, 4.5, 4.8, 5.0, 5.7, 5.8, 6.0, 6.1, 6.2, 6.5, 6.6, 6.8, 7.0, 7.5, 8.2, 8.8, 9.0]
blood_glucose_level: [80.0, 85.0, 90.0, 100.0, 126.0, 130.0, 140.0, 145.0, 155.0, 158.0, 159.0, 160.0, 200.0, 220.0, 240.0, 260.0, 280.0, 300.0]
So gia tri phan biet cua age (doi chieu): 102


Xử lý:
Giữ, ghi nhận hạn chế. Chỉ số đo liên tục mà chỉ 18 giá trị cố định (age có hơn 100) là dấu hiệu dữ liệu sinh tổng hợp, không đo trên bệnh nhân thật. Không dùng làm căn cứ loại dòng.

### Biến hạng mục: giá trị và tần suất

In [23]:
for c in ["gender", "smoking_history"]:
    print(f"{c}:")
    dem = Counter(r[c] for r in rows)
    for gia_tri, so_luong in sorted(dem.items(), key=lambda t: -t[1]):
        print(f"  {gia_tri:<14}: n={so_luong:>6}  ({100.0*so_luong/len(rows):>5.2f}%)")
    print()

gender:
  Female        : n= 58552  (58.55%)
  Male          : n= 41430  (41.43%)
  Other         : n=    18  ( 0.02%)

smoking_history:
  No Info       : n= 35816  (35.82%)
  never         : n= 35095  (35.09%)
  former        : n=  9352  ( 9.35%)
  current       : n=  9286  ( 9.29%)
  not current   : n=  6447  ( 6.45%)
  ever          : n=  4004  ( 4.00%)



Xử lý:
gender="Other" (18 dòng, 0,018%): giữ, one-hot riêng - mẫu nhỏ nhưng không tự ý gộp/loại.
smoking_history="No Info" (35.816 dòng, 35,8%): giữ thành nhóm riêng - thiếu ngầm nhưng chiếm quá lớn để loại.

## 4. Tỷ lệ mắc theo từng chỉ số

In [24]:
tuoi = lambda r: float(r["age"])
in_bang("Tuoi (nguong ADA/CDC Diabetes Risk Test):", [
    ("<40",   [r for r in rows if tuoi(r) < 40]),
    ("40-49", [r for r in rows if 40 <= tuoi(r) < 50]),
    ("50-59", [r for r in rows if 50 <= tuoi(r) < 60]),
    (">=60",  [r for r in rows if tuoi(r) >= 60]),
])
print()

hba1c = lambda r: float(r["HbA1c_level"])
in_bang("HbA1c (nguong ADA):", [
    ("<5.7",    [r for r in rows if hba1c(r) < 5.7]),
    ("5.7-6.4", [r for r in rows if 5.7 <= hba1c(r) < 6.5]),
    (">=6.5",   [r for r in rows if hba1c(r) >= 6.5]),
])
print()

glu = lambda r: float(r["blood_glucose_level"])
in_bang("Glucose (nguong ADA):", [
    ("<100",    [r for r in rows if glu(r) < 100]),
    ("100-125", [r for r in rows if 100 <= glu(r) < 126]),
    (">=126",   [r for r in rows if glu(r) >= 126]),
])
print()

bmi = lambda r: float(r["bmi"])
in_bang("BMI (nguong WHO):", [
    ("<18.5",     [r for r in rows if bmi(r) < 18.5]),
    ("18.5-24.9", [r for r in rows if 18.5 <= bmi(r) < 25]),
    ("25-29.9",   [r for r in rows if 25 <= bmi(r) < 30]),
    (">=30",      [r for r in rows if bmi(r) >= 30]),
])

Tuoi (nguong ADA/CDC Diabetes Risk Test):
  <40         : n= 45487  ty le mac=  1.56%
  40-49       : n= 14595  ty le mac=  6.84%
  50-59       : n= 14863  ty le mac= 12.51%
  >=60        : n= 25055  ty le mac= 19.68%

HbA1c (nguong ADA):
  <5.7        : n= 37857  ty le mac=  0.00%
  5.7-6.4     : n= 41346  ty le mac=  8.00%
  >=6.5       : n= 20797  ty le mac= 24.96%

Glucose (nguong ADA):
  <100        : n= 21119  ty le mac=  0.00%
  100-125     : n=  7025  ty le mac=  0.00%
  >=126       : n= 71856  ty le mac= 11.83%

BMI (nguong WHO):
  <18.5       : n=  8494  ty le mac=  0.75%
  18.5-24.9   : n= 22219  ty le mac=  3.88%
  25-29.9     : n= 45751  ty le mac=  7.30%
  >=30        : n= 23536  ty le mac= 17.99%


In [25]:
in_bang("Cao huyet ap:", [
    ("khong", [r for r in rows if r["hypertension"] == "0"]),
    ("co",    [r for r in rows if r["hypertension"] == "1"]),
])
print()

in_bang("Benh tim:", [
    ("khong", [r for r in rows if r["heart_disease"] == "0"]),
    ("co",    [r for r in rows if r["heart_disease"] == "1"]),
])
print()

in_bang("Gioi tinh:", [
    (g, [r for r in rows if r["gender"] == g])
    for g in sorted(set(r["gender"] for r in rows))
])
print()

in_bang("Hut thuoc:", [
    (h, [r for r in rows if r["smoking_history"] == h])
    for h in sorted(set(r["smoking_history"] for r in rows))
])

Cao huyet ap:
  khong       : n= 92515  ty le mac=  6.93%
  co          : n=  7485  ty le mac= 27.90%

Benh tim:
  khong       : n= 96058  ty le mac=  7.53%
  co          : n=  3942  ty le mac= 32.14%

Gioi tinh:
  Female      : n= 58552  ty le mac=  7.62%
  Male        : n= 41430  ty le mac=  9.75%
  Other       : n=    18  ty le mac=  0.00%

Hut thuoc:
  No Info     : n= 35816  ty le mac=  4.06%
  current     : n=  9286  ty le mac= 10.21%
  ever        : n=  4004  ty le mac= 11.79%
  former      : n=  9352  ty le mac= 17.00%
  never       : n= 35095  ty le mac=  9.53%
  not current : n=  6447  ty le mac= 10.70%


### Cấu trúc nhãn

Bảng trên: tỷ lệ mắc 0.00% ở HbA1c<5.7 và glucose<126. Kiểm tra chính xác, không làm tròn.

In [26]:
mac = [r for r in rows if r["diabetes"] == "1"]
ngoai_cong = [r for r in mac if not (hba1c(r) >= 5.7 and glu(r) >= 126)]
print("So ca mac nam NGOAI ca hai nguong (HbA1c>=5.7 va glucose>=126):", len(ngoai_cong))
print("HbA1c nho nhat trong nhom mac  :", min(hba1c(r) for r in mac))
print("Glucose nho nhat trong nhom mac:", min(glu(r) for r in mac))

# Trong so nguoi da qua ca hai nguong, con bao nhieu % thuc su mac?
qua_cong = [r for r in rows if hba1c(r) >= 5.7 and glu(r) >= 126]
print(f"\nSo nguoi qua ca hai nguong: {len(qua_cong)} ({100.0*len(qua_cong)/len(rows):.2f}% du lieu)")
print(f"Trong do ty le mac thuc su : {ti_le_mac(qua_cong):.2f}%")

So ca mac nam NGOAI ca hai nguong (HbA1c>=5.7 va glucose>=126): 0
HbA1c nho nhat trong nhom mac  : 5.7
Glucose nho nhat trong nhom mac: 126.0

So nguoi qua ca hai nguong: 45608 (45.61% du lieu)
Trong do ty le mac thuc su : 18.64%


Xử lý:
Cả 8.500 ca mắc đều thỏa HbA1c>=5.7 và glucose>=126, không ngoại lệ - điều kiện cần.
Chưa đủ: 45,61% dữ liệu qua cả hai ngưỡng nhưng chỉ 18,64% trong đó mắc bệnh. Cần tổ hợp thêm tuổi, BMI, cao huyết áp, bệnh tim: logistic regression, không phải ngưỡng đơn hay cộng trung bình prevalence.

Tổng hợp mục 4: tỷ lệ mắc tăng dần theo HbA1c, glucose, tuổi, BMI; cao huyết áp và bệnh tim tăng khoảng 4 lần. Không chỉ số đơn lẻ nào quyết định nhãn (mục trên) nên chọn logistic regression để tổ hợp nhiều yếu tố.

## 5. Tổng hợp tiền xử lý

| Vấn đề | Xử lý | Lý do |
|---|---|---|
| Trùng lặp 3.854 dòng | Loại, còn 96.146 | Không thêm thông tin |
| Mất cân bằng 91.5/8.5 | Đánh giá bằng precision/recall/F1 | Accuracy không phản ánh nhóm bệnh |
| age<1 (911 dòng, tuổi tính theo tháng) | Giữ nguyên | Tuổi hợp lệ; loại là can thiệp chủ quan |
| bmi=27.32 lặp lại (25.495 dòng, 25,5%) | Giữ nguyên, ghi nhận hạn chế | Bị điền khuyết bằng mean từ trước, không khôi phục được |
| HbA1c/glucose chỉ 18 giá trị phân biệt | Ghi nhận hạn chế (dữ liệu tổng hợp) | Không dùng làm căn cứ loại dòng |
| gender="Other" (18 dòng, 0,018%) | Giữ, one-hot riêng | Mẫu nhỏ nhưng không tự ý gộp/loại |
| smoking_history="No Info" (35.816 dòng, 35,8%) | Giữ thành nhóm riêng | Thiếu ngầm nhưng chiếm quá lớn để loại |
| Biến hạng mục (gender, smoking) | One-hot | Không gán thứ hạng số tùy tiện |
| Đặc trưng số | Chuẩn hóa theo train | Ổn định khi huấn luyện logistic |